## Clase 4 - Funciones robustas, manejo de errores y excepciones

### 1. ¿Por qué esta clase es importante?

En programación para ciencia de datos, muchas veces los errores no aparecen porque el programador "no sabe programar" sino porque los datos reales son imperfectos.

In [95]:
# En un curso introductorio, normalmente trabajamos con ejemplos límpios:
edades = [20, 25, 30, 35]
edades

[20, 25, 30, 35]

In [96]:
# Pero en un problema real podríamos recibir algo así:
edades = [20, 25, None, "treinta", -5, 200, 35]
edades

[20, 25, None, 'treinta', -5, 200, 35]


Aquí aparecen varios problemas:

- None          # dato faltante
- "treinta"     # texto donde esperábamos un número
- -5            # edad inválida
- 200           # valor fuera de rango

Si nuestro código no está preparado, el programa puede fallar o, peor aún, puede producir resultados incorrectos sin avisarnos.

### 2. De funciones reutilizables a funciones robustas

En la clase 3 vimos que las funciones ayudan a evitar código repetitivo.

In [97]:
# Ejm: si queremos calcular un promedio:
notas = [15, 28, 12, 20]

total = sum(notas)
promedio = total / len(notas)

print(promedio)

18.75


In [98]:
# Esta version funciona, pero está escrita directamente en el programa principal.
# Una mejor versión es usar una función:
def calcular_promedio(valores):
    total = sum(valores)
    promedio = total / len(valores)
    return promedio

notas = [15, 18, 12, 20]
print(calcular_promedio(notas))

16.25


Esto y es mejor porque podemos reutilizar la función.

Pero todavía hay un poblema: la función asume que todo está bien.

¿Qué pasa si la lista está vacía?

In [99]:
notas = []
print(calcular_promedio(notas))

ZeroDivisionError: division by zero

Aparece un error:

ZeroDivisionError: division by zero

In [ ]:
# Qué pasa si la lista tiene texto?
notas = [15, 18, "aprobado", 20]
print(calcular_promedio(notas))

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Aparece un error:

TypeError

**Entonces, una dunción no solo debe ser reutilizable. También debe ser segura.**

### 3. Validaciones dentro de funciones

3.1 ¿Qué es validar?

Validar significa verificar si los datos cumplen las condiciones mínimas antes de procesarlos.

Por ejemlo, antes de calcular el promedio de una lista, deberíamos verificar que:
- El dato sea una lista.
- La lista no esté vacía.
- Todos los elementos sean numéricos.
- No haya valores imposibles o fuera del dominio del problema.

En ciencia de datos, validar datos es fundamental porque los modelos, reportes y análisis dependen de la calidad de la información de entrada.

#### 3.2 caso inicial: calcular promedio sin función

Supongamos que tenemos notas de estudiantes.

In [ ]:
notas = [15, 18, 12, 20]

total = 0

for nota in notas:
    total += nota

promedio = total / len(notas)

print("Promedio", promedio)

Promedio 16.25


Este código funciona si todas la snotas son correctas.

Ahora probemos con datos sucios:


In [ ]:
notas = [15, 18, "ausente", 20]

total = 0

for nota in notas:
    total += nota
    
promedio = total / len(notas)

print("Promedio;", promedio)

# El error sería de tipo

TypeError: unsupported operand type(s) for +=: 'int' and 'str'

#### 3.3 ¿Por qué falla este código?

El programa falla porque Python no puede ejecutar esta operación
- 18 + "ausente"

Python sí sabe sumar números:
- 10 + 5

Pero no sabe sumar directamente un número con un texto.
- 18 + "ausente"

Este es un error en tiempo de ejecusión. El programa está bien escrito sintácticamente, pero falla cuando se ejecuta.

#### 3.4 Versión con función simple

Ahora llevemos el cálculo a una función, como se trabajó en la clase anterior.

In [ ]:
def calcular_promedio(notas):
    total = 0
    
    for nota in notas:
        total += nota
        
    promedio = total / len(notas)
    return promedio

notas = [15, 18, 12, 20]
resultado = calcular_promedio(notas)

print("Promedio:" , resultado)

Promedio: 16.25


Esta función es reutilizable, pero todavía no es robusta.

Si le ponemos datos incorrectos:

In [ ]:
notas = [15, 18, "ausente", 20]
resultado = calcular_promedio(notas)

print("Promedio", resultado)

TypeError: unsupported operand type(s) for +=: 'int' and 'str'

El programa vuleve a fallar:

Entonces necesitamos mejorar la función.

#### 3.5 Primera mejora: Válidar manualmente los datos

Podemos verificar si cada nota es numérica antes de sumarla.

In [ ]:
def calcular_promedio_validar(notas):
    total = 0
    cantidad = 0
    
    for nota in notas:
        if isinstance(nota, (int, float)):
            total += nota
            cantidad += 1
            
    promedio = total / cantidad
    return promedio

notas = [15, 18, "ausente", 20]
resultado = calcular_promedio_validar(notas)

print("Promedio;", resultado)

Promedio; 17.666666666666668


#### 3.6 Problema de ignorar errores silenciosamente
 Cuando una función omite datos incorrectos sin avisar, puede generar resultados engañosos.

In [ ]:
# Ejm:
notas = [15, "error", None]
notas

[15, 'error', None]

Con la función anterior, el promedio se calcularía solo con 15.

Eso puede dar una falsa sensación de que todo está bien.

Por eso, en programación profesional no basta con evitar que el programa se rompa. También debemos informar qué problema ocurrió.

Aquí entra el manejo de errores y excepciones.

### 4. Introducción a errores y excepciones
#### 4.1 ¿Qué es un error?

Un error es una situación que impide que el programa funcione correctamente.

En Python podemos distinguir tres tipos importantes:

a) Error sintántico
- Ocurre cuando el código está mal escrito según las reglas del lenguaje.

In [ ]:
# EJm:
if True
    print("Hola")

SyntaxError: expected ':' (3317139496.py, line 2)

In [ ]:
# Forma correcta es:
if True:
    print("Hola")

Hola


b) Error en tiempo de ejecusión
- Ocurre cuando el código está bien escrito, pero falla al ejecutarse.

In [ ]:
# Ejm:
print(10 / 0)
# Este código está bien escrito, pero falla porque no se puede dividir entre cero.

ZeroDivisionError: division by zero

c) Error semántico
- Ocurre cuando el programa no se rompe, pero produce un resultado incorrecto.

In [ ]:
#Ejm:
# Queremos sumar del 1 al 5:

total = 0

for i in range(5):
    total += i
    
pritn(total)

NameError: name 'pritn' is not defined

In [ ]:
# Pero si queremos sumar:
1 + 2 + 3+ 4 +5

# El resultado correcta era:


15

In [ ]:
# El error está en que range(5) genera:
0, 1, 2, 3, 4

(0, 1, 2, 3, 4)

In [ ]:
# La versión cortrecta sería:
total = 0

for i in range(1, 6):
    total += i
    
print(total)

15


#### 4.2 ¿Qué es una excepción?

Una excepción es un evento que ocurre durante la ejecución del programa y que interrumpe su flujo normal.

In [ ]:
# EJm.
notas = [15, 18, "audente", 20]

total = 0

for nota in notas:
    total += nota


TypeError: unsupported operand type(s) for +=: 'int' and 'str'

Cuando Python llega a "ausente", no puede sumarlo al total y lanza una excepción.

La excepción indica que ocurrió algo inesperado.

#### 4.3 Excepciones comunes en Python

Algunas excepciones frecuentes son:
- ZeroDivisionError : Aparece cuando dividimos entre cero.
- TypeError : Aparece cuando aplicamos una operación a un tipo de dato incorrecto.
- ValueError : Aparece cuando el tipo de dato es correcto, pero el valor no es válido.
- IndexError : Aparece cuando intentamos acceder a una posición inexistente en una lista.
- KeyError : Aparece cuando intentamos acceder a una clave inexistente en un diccionario.
-NameError : Aparece cuando usamos una variable o función que no existe.

### 5. Manejo de excepciones con try y except
#### 5.1 ¿Para qué sirve try?

try permite ejecutar un bloque de código que podría generar un error.

#### 5.2 ¿Para qué sirve except?

except permite capturar y manejar el error para que el programa no se detenga abruptamente.

La estructura básica es:

try:

   *# código que puede fallar*

except:

   *# código que se ejecuta si ocurre un error*

#### 5.3 Ejemplo simple sin manejo de errores

In [ ]:
numero = int(input("Ingrese un número: "))
print("Número ingresado:", numero)

Número ingresado: 4


Si el usuario escribe:
- 10

pero si escribe:
- hola

El programa falla con.
- ValueError

#### 5.4 Ejemplo con manejo de errores

In [ ]:
numero = "hola"

try:
    numero = int(numero)
    print("Número ingresado.", numero)
    
except ValueError:
    print("Error: debe ingresar un número entero.")
    
    # Ahora el programa no se rompe. En lugar de eso, muestra un mensaje claro.

Error: debe ingresar un número entero.


#### 5.5 Aplicación al cálculo de promedio

Ahora mejoremos nuestra función.

In [100]:
def calcular_promedio_seguro(notas):
    total = 0
    cantidad = 0
    
    for nota in notas:
        try:
            total += float(nota)
            cantidad += 1
        
        except ValueError:
            print(f"Valor omitido porque no puede convertirse a número: {nota}")
        
        except TypeError:
            print(f"Valor omitido porque tiene un tipo inválido: {nota}")
            
    promedio = total / cantidad
    return promedio

In [101]:
# Probemos
notas = [15, 18, "ausente", None, 20]

resultado = calcular_promedio_seguro(notas)

print(resultado)

Valor omitido porque no puede convertirse a número: ausente
Valor omitido porque tiene un tipo inválido: None
17.666666666666668


#### 5.6 ¿Qué hemos ganado?
Antes, el programa se rompía.

Ahora:
- El programa continúa.
- Se informa qué dato fue omitido.
- Se calcula el promedio con los datos válidos.
- El código queda encapsulado en una función reutilizable.

Pero todavía falta mejorar algo.

¿Qué pasa si todos los datos son inválidos?

In [ ]:
notas = ["ausente", None, "error"]


En este caso, la cantidad queda en cero y ocurre:
- ZeroDivisionError

Entonces necesitmos seguir haciendo la función más robusta.

### 6. Uso de raise para controlar errores

#### 6.1 ¿Qué es raise?
raise sirve para lanzar un error manualmente.

Se usa cuando queremos detener una función porque los datos no cumplen una condición necesaria.

In [111]:
def validar_nota(nota):
    if nota < 0 or nota > 20:
        raise ValueError("La nota debe estar entre 0 y 20.")
    
    return True

Aquí estamos diciendo.

**Si la nota no está entre 0 y 20, este dato no es válido y debe lanzarse un error.**

In [113]:
validar_nota(12)

True

In [114]:
validar_nota(30)

ValueError: La nota debe estar entre 0 y 20.

#### 6.2 ¿Por qué usar raise?

Porque hay errores que no deberían ser ignorados.

Por ejemplo, una nota de 25 en un sistema de evaluación sobre 20 no debe aceptarse.

In [115]:
nota = 25

Ese valor no es un problema de Python. Python puede operar con el número 25.

El problema es del dominio del negocio o del análisis.

Por eso usamos raise.

#### 6.3 Función para validar una nota

In [116]:
def validar_nota(nota):
    if not isinstance(nota, (int, float)):
        raise TypeError("La nota debe ser numérica.")
    
    if nota < 0 or nota > 20:
        raise ValueError("La nota debe estar entre 0 y 20.")
    
    return True

In [117]:
# Probemos:
validar_nota(18)

True

In [118]:
# Ahora:
validar_nota(25)

ValueError: La nota debe estar entre 0 y 20.

In [119]:
# Ahora:
validar_nota("ausente")

TypeError: La nota debe ser numérica.

#### 6.4 Versión sin función

Antes de usar funciones robustas, podríamos escribir todo directamente:

In [120]:
notas = [15, 18, "ausente", None, 20]

notas_validas = []

for nota in notas:
    try:
        if not isinstance(nota, (int, float)):
            raise TypeError("La nota debe ser numérica.")
        
        if nota < 0 or nota > 20:
            raise ValueError("La nota debe estar entre 0 y 20.")
        
        notas_validas.append(nota)
        
    except TypeError as error:
        print(f"Dato omitido: {nota}. Motivo: {error}")
    
    except ValueError as error:
        print(f"Datos omitido: {nota}. Motivo: {error}")
        
print("Notas válidas:", notas_validas)

Dato omitido: ausente. Motivo: La nota debe ser numérica.
Dato omitido: None. Motivo: La nota debe ser numérica.
Notas válidas: [15, 18, 20]


Esta versión funciona, pero el bloque de validación está mezclado con el programa principal.

Si tenemos que validar notas en varias partes del notebook, repetiríamos el mismo código muchas veces.

#### 6.5 Versión con función

Ahora separamos la validación en una función.

In [121]:
def validar_nota(nota):
    if not isinstance(nota, (int, float)):
        raise TypeError("La nota debe ser numérica.")
    
    if nota < 0 or nota > 20:
        raise ValueError("La nota debe estar entre 0 y 20.")
    
    return True

notas =[15, 18, 25, "ausente", None, 20]

notas_validas = []

for nota in notas:
    try:
        validar_nota(nota)
        notas_validas.append(nota)
        
    except TypeError as error:
        print(f"Dato omitido: {nota}. Motivo: {error}")
    
    except ValueError as error:
        print(f"Dato omitido: {nota}. Motivo: {error}")
        
print("Notas válidas:", notas_validas)

Dato omitido: 25. Motivo: La nota debe estar entre 0 y 20.
Dato omitido: ausente. Motivo: La nota debe ser numérica.
Dato omitido: None. Motivo: La nota debe ser numérica.
Notas válidas: [15, 18, 20]


Esta versión es mejor porque:
- La validación está encapsulada.
- El código principal es más claro.
- la función validar_nota() se puede reutilizar.
- Los errores tienen mensajes claros.
- El programa no se detiene ante el primer dato incorrecto

### 7. Función robusta completa para calcular el promedio

Ahora usamos todo.

In [123]:
def validar_nota(nota):
    if not isinstance(nota, (int, float)):
        raise TypeError("La nota debe ser numérica")
    
    if nota < 0 or nota > 20:
        raise ValueError("La nota debe estar entre 0 y 20")
    
def calcular_promedio_robusto(notas):
    notas_validas = []
    
    for nota in notas:
        try:
            validar_nota(nota)
            notas_validas.append(nota)
            
        except TypeError as error:
            print(f"Dato omitido: {nota}. Motivo: {error}")
            
        except ValueError as error:
            print(f"Dato omitido: {nota}. Motivo: {error}")
            
    if len(notas_validas) == 0:
        raise ValueError("No existe notas válidas para calcular el promedio.")
            
    promedio = sum(notas_validas) / len(notas_validas)
    return promedio

In [124]:
# Probemos:
notas = [15, 18, 25, "ausente", None, 20]

resultado = calcular_promedio_robusto(notas)

print("Promedio final", resultado)

Dato omitido: 25. Motivo: La nota debe estar entre 0 y 20
Dato omitido: ausente. Motivo: La nota debe ser numérica
Dato omitido: None. Motivo: La nota debe ser numérica
Promedio final 17.666666666666668


#### 7.1. ¿Qué pasa si todos los datos son inválidos?

In [125]:
notas = ["ausente", None, 30, -5]

resultado = calcular_promedio_robusto(notas)

print("Promedio final.", resultado)

Dato omitido: ausente. Motivo: La nota debe ser numérica
Dato omitido: None. Motivo: La nota debe ser numérica
Dato omitido: 30. Motivo: La nota debe estar entre 0 y 20
Dato omitido: -5. Motivo: La nota debe estar entre 0 y 20


ValueError: No existe notas válidas para calcular el promedio.

Aquí la función lanza:

ValueError: No xiste notas válidas para calcular el promedio.

Este comportamiento es correcto porque no tiene sentido calcular un promedio sin datos válidos.

### 8. Uso de else y finally
#### 8.1. ¿Para qué sirve else?

En un bloque try/except, else se ejecuta solo si no ocurrió ningún error.

Estructura:

In [ ]:
try:
    # Código que puede fallar
except:
    # Código si ocurre error
else:
    # Código si no ocurre error

#### 8.2. ¿Para que sirve finally?

finally se ejecuta siempre, ocurra o no ocurra un error.

Estructura:

In [ ]:
try:
    # Código que puede fallar
except:
    # Código si ocurre un error
else:
    # Código si no ocurre un error
finally:
    # Código que se siempre se ejecuta

Se usa mucho para cerrar archivos, cerrar conexiones, liberar recursos o dejar mensajes de seguimiento.

#### 8.3. Ejemplo didáctico


In [128]:
numero = "hola"

try:
    numero = int(numero)

except ValueError:
    print("Error: el valor ingresado  no es un número.")

else:
    print("Conversión exitosa.")
    print("Número ingresado.", numero)

finally:
    print("Fin del proceso")

Error: el valor ingresado  no es un número.
Fin del proceso


#### 8.4 Aplicación al promedio robusto

In [131]:
notas = [15, 18, 25, "ausente", None, 20]

try:
    resultado = calcular_promedio_robusto(notas)

except ValueError as error:
    print("No se pudo calcular el promedio.")
    print("Morivo:", error)

else:
    print("El promedio fue calculado correctamente.")
    print("Promedio final", resultado)
    
finally:
    print("Proceso de análisis de notas finalizado.")

Dato omitido: 25. Motivo: La nota debe estar entre 0 y 20
Dato omitido: ausente. Motivo: La nota debe ser numérica
Dato omitido: None. Motivo: La nota debe ser numérica
El promedio fue calculado correctamente.
Promedio final 17.666666666666668
Proceso de análisis de notas finalizado.
